# 🧊 3D Pose Estimation with DeepLabCut

This notebook documents the process of using DeepLabCut for **3D pose estimation** from two synchronized camera videos. It includes:

- Creating a 3D project
- Calibrating the cameras using checkerboard images
- Verifying calibration through undistortion
- Triangulating 2D keypoints into 3D coordinates
- Creating labeled 3D visualizations

The 3D pipeline enables accurate tracking of body part movements in space and is particularly useful for behavior analysis and neuroscience experiments.

## 📁 Step 1: Create a 3D DeepLabCut Project

In [ ]:
import deeplabcut

# Create a new 3D project
config_path3d = deeplabcut.create_new_project_3d(
    "My3DProject", "YourName", num_cameras=2,
    working_directory="/path/to/project"
)

This command initializes a 3D project directory with subfolders for calibration, camera matrices, corners, and undistortion. Ensure your 2D project(s) which you got from napari are ready beforehand.

## 📸 Step 2: Capture and Process Calibration Images

In this step, we capture images of a printed checkerboard from both cameras to calibrate the 3D setup. These images will be used to estimate the intrinsic and extrinsic parameters of each camera.

###  What You Need:
- A **printed checkerboard** (preferably 8×6 squares)
  - Download templates from: [https://markhedleyjones.com/projects/calibration-checkerboard-collection](https://markhedleyjones.com/projects/calibration-checkerboard-collection)
- Two **synchronized cameras** in a fixed stereo configuration.
- Optionally: short videos instead of individual snapshots.
- An example video you can find in the folder of the project by the name (trial0001.mp4).

###  Option A: Capture Synchronized Images
Take still images from both cameras simultaneously. Save them using the following naming convention:
camera-1-01.jpg camera-2-01.jpg camera-1-02.jpg camera-2-02.jpg ...


You should aim for at least **30–70 pairs** for reliable calibration.



### 🎞️ Option B: Extract Frames from Videos
If you recorded videos instead of taking photos, extract frames using `ffmpeg` in your terminal:

In [ ]:
ffmpeg -i camera1_video.mp4 -vframes 30 camera-1-%03d.jpg
ffmpeg -i camera2_video.mp4 -vframes 30 camera-2-%03d.jpg

%03d ensures filenames like camera-1-001.jpg, camera-2-001.jpg, etc.

Make sure that matching numbers are temporally aligned!

### 📁 Image Placement
Move all image pairs to:

/Your3DProject/calibration_images/

### ✅ Tips for Good Calibration Images
Keep checkerboard flat and visible in both cameras.

Don't rotate the checkerboard more than ~30°.

Cover multiple distances, angles, and positions in the image.

Use consistent lighting.

## 🔧 Step 3: Calibrate the Cameras

Now that we have checkerboard image pairs from both cameras, we can begin calibrating them. Calibration calculates both **intrinsic** (lens) and **extrinsic** (position/orientation) parameters for each camera.

###  First Pass: Visualize and Clean Up
This step will detect checkerboard corners and save visualizations to the `corners/` folder. Set `calibrate=False` to allow manual review.


In [ ]:
deeplabcut.calibrate_cameras(config_path3d, cbrow=8, cbcol=6, calibrate=False, alpha=0)

cbrow=8 and cbcol=6 refer to the number of the inner corners not squares. In other words, number of rows and columns minus 1

alpha=0 minimizes black borders in the undistorted images. you can change it as what provides better results.

Review the outputs in the corners/ folder.  you dont need to remove any image pairs where corners are missing or misaligned now because the command already moves them to a folder called removed_calibration_images. If you can't find it in the project folder you created, create one and move the unwanted pairs.

example images from the corner file where the code detects the corners are in the project folder, check them!

## 🧮 Final Calibration
Once you have a clean set of image pairs, set calibrate=True to perform the full calibration:

In [ ]:
deeplabcut.calibrate_cameras(config_path3d, cbrow=8, cbcol=6, calibrate=True, alpha=0)

This will:

- Estimate the intrinsic/extrinsic parameters

- Perform stereo rectification

- Save results to camera_matrix/stereo_params.pickle

### 🔍 Understanding `calibrate=True` vs `False`

The `deeplabcut.calibrate_cameras` function accepts a `calibrate` argument that controls what the function does:

#### 🛠️ `calibrate=False`:
- Detects checkerboard corners in each image pair.
- Saves visualizations in the `corners/` folder.
- Allows you to inspect and **manually remove bad image pairs**.
- Use this to clean the data **before** actual calibration.

#### 🛠️ `calibrate=True`:
- Performs the full stereo calibration.

- Computes:

- Intrinsic and extrinsic parameters for both cameras.

- Stereo rectification between views.

- Saves results to camera_matrix/stereo_params.pickle.


💡 Always run calibrate=False first to inspect and clean your images, then run calibrate=True for final calibration.

## 👓 Step 4: Check Undistortion

After calibration, we need to verify how well the stereo calibration worked. This is done by **undistorting** the original calibration images and projecting the detected corner points on top of them.

This step will:
- Undistort all calibration images using the computed camera parameters.
- Overlay corner detections on the undistorted images.
- Save the results to the `undistortion/` folder.
- Show a 3D scatter plot of triangulated corner points to visualize alignment.



### 📌 Run the Check

In [ ]:
deeplabcut.check_undistortion(config_path3d, cbrow=8, cbcol=6)

## 🔍 What to Look For
- In the undistortion/ folder, you'll find undistorted image pairs with corner points overlaid.

- Points should accurately align with the checkerboard pattern.

- The 3D scatter plot should show a smooth, grid-like distribution of the corners.

If things look off (misaligned dots, scattered 3D grid), you should:

- Go back to Step 3.

- Remove bad calibration image pairs.

- Rerun calibration with calibrate=True.

## 🧠 Step 5: Triangulate 2D Keypoints into 3D

Now that the cameras are calibrated and undistortion is verified, we can **triangulate** the 2D keypoints into 3D space.

This step uses:
- The 2D pose estimation results from both cameras
- The stereo calibration parameters
- Video files from both cameras with proper naming

---

### ⚠️ Requirements Before You Start

1. Make sure your 2D videos are analyzed and filtered.
2. Both videos must follow a consistent naming convention like: rig1_mouse_camera-1.avi rig1_mouse_camera-2.avi \
3. The folder that stores the original video you want to triangulate, must also inculde all the files you got from napari for both cameras: the pickle, meta pickle, h5 and excel (or csv) files. They must be moved from the 2D project folders created by napari for both cameras to this folder before triangulate.
4. Edit the 3D config.yaml file (see example).
#### Important: The camera names in the filenames must match those in the 3D `config.yaml`

3. The 3D `config.yaml` must include:
- Paths to 2D projects
- Body part names (as in the 2D yaml of the 2D videos)
- Snapshot index (e.g. `-1` for last trained model) (as in the yaml of the 2D videos)
- `skeleton` definition (optional, used only for plotting) (as in the yaml of the 2D videos)

note:  
In the folder there is a 3D config.yaml example file.

In [ ]:
deeplabcut.triangulate(
 config_path3d,
 video_path="/path/to/folder/with/videos",
 filterpredictions=True)

filterpredictions=True will apply a median filter to smooth noisy 2D predictions (recommended).

Output .h5 and .csv files will be saved in the same folder as the input videos.

## 📁 Output
You will get a file like:

"YourVideoName_DLC_3D_ScorerName.h5"

This file contains 3D coordinates (x, y, z) for each labeled body part across frames.

## 🎥 Step 6: Create a 3D Labeled Video

Now that we’ve triangulated the 3D pose data, we can create a **3D labeled video** to visualize body part movements in space.

This step generates a 3D animation by:
- Loading the triangulated `.h5` file
- Plotting the 3D positions frame by frame
- Saving the video as a sequence of `.png` images (and combining them)

---

### ▶️ Create the Labeled 3D Video

In [ ]:
deeplabcut.create_labeled_video_3d(
    config_path3d,
    triangulated_file_folder=["/path/to/folder/with/triangulated/files"],
    start=0,
    end=300,
    xlim=[-50, 50],
    ylim=[-50, 50],
    zlim=[0, 100],
    view=[113, 270],
    draw_skeleton=True
)

- start and end define the range of frames to plot.

- xlim, ylim, zlim control the 3D plot axis limits — adjust based on your data.

- view controls the 3D plot angle (elevation, azimuth).

- draw_skeleton=True will connect the points visually based on your skeleton definition in the config file.

#### 💡 Tip: Automatically Set Axis Limits You can calculate the xlim, ylim, and zlim ranges directly from your triangulated data:

In [ ]:
# Calculate min and max for each coordinate across all body parts
x_min, x_max = df_3d.xs('x', level='coords', axis=1).min().min(), df_3d.xs('x', level='coords', axis=1).max().max()
y_min, y_max = df_3d.xs('y', level='coords', axis=1).min().min(), df_3d.xs('y', level='coords', axis=1).max().max()
z_min, z_max = df_3d.xs('z', level='coords', axis=1).min().min(), df_3d.xs('z', level='coords', axis=1).max().max()

Then use these as:

In [ ]:
xlim=[x_min, x_max], ylim=[y_min, y_max], zlim=[z_min, z_max]

## ⚙️ Optional Parameters You Can Add
- trailpoints: number of frames to leave a motion trail

- figsize: size of the 3D figure (e.g., (80, 8))

- fps: frames per second of the output video

- dpi: resolution of the output

## 📁 Output
This command creates:

- A temporary folder of .png images

- A .mp4 labeled 3D video (based on those frames)

You can use this to visually inspect pose quality and behavior patterns in 3D.

**In the project folder, you can find an example code for the whole progress, in this code i used the main commands and add functions that helped in enhancing the extracted images to help the model of deeplabcut find the corners and learn better, however it's optional to use them and depends on the user's images and case.**